# Shared representation at the second-number position — paper figures

Qwen2.5-7B-Instruct, `max(a, b)` one-shot prompt. Loads `results/sharedrep_*.npz` written by
`data_sharedrep.py`. L13 directions come from `results/numberreps_*.npz`.

Both directions are read at the second-number position, so both are $\mathbf{v}$'s and the
subscript says which number each carries: $\mathbf{v}_1$ = H14 DAS 1D, inside head
L14.H14's output, written in at $n_2$ · $\mathbf{v}_2$ = L13 PC1 at $n_2$, already in place.
The first number's direction at its *own* position is $\mathbf{u}$, in
`makefig_numberreps.ipynb`.

In [ ]:
import os, json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers the 3d projection)

RESULTS_DIR, FIGS_DIR = 'results', 'figs'
os.makedirs(FIGS_DIR, exist_ok=True)

REGIME = '2digit'
D    = dict(np.load(f'{RESULTS_DIR}/sharedrep_{REGIME}.npz'))
MJ   = json.load(open(f'{RESULTS_DIR}/sharedrep_meta.json'))['regimes'][REGIME]
M    = MJ['meta']

a_vals, b_vals = D['a_vals'], D['b_vals']
print(f"{REGIME}: L{M['l13_layer']} -> L{M['h_layer']}.H{M['h_head']}   "
      f"n1 tok {M['pos_first']}, n2 tok {M['pos_second']}   "
      f"cloud n={M['n_cloud']}  eval n={M['n_eval']}  case '{M['case']}'")
print(f"H14 DAS reachable through W_O^h: {M['h14_das_reachable']:.3f}")
print(f"probe log A held-out R2 = {MJ['probe_logA_stats']['r2_test']:.3f}   "
      f"probe log B held-out R2 = {MJ['probe_logB_stats']['r2_test']:.3f}")

## Plot style

In [ ]:
LABEL_FS, TICK_FS, LEGEND_FS, TITLE_FS = 18, 14, 15, 19
DPI, SAVE = 300, True

# Same validated categorical palette as makefig_numberreps.ipynb (CVD-checked; no red/green pair),
# plus one teal for the H14 scope.
COL = {'full': '#2166AC',   # blue    -> full-residual scopes
       'das':  '#C97B00',   # amber   -> L13 DAS / arm 2
       'pc1':  '#C06A9B',   # orchid  -> L13 PC1 / arm 3
       'h14':  '#117777',   # teal    -> H14 scopes
       'ceil': '#6E6E6E'}   # grey    -> the L14-full ceiling
JOINT_HATCH = '//'          # joint (H14 + L13) conditions are the same colour, hatched

# One ramp per number, with the HUE carrying which number it is and lightness its magnitude:
# y1 warm, y2 cool. Both are ColorBrewer sequential maps (monotonic lightness, so the ordering
# survives greyscale), and warm-vs-cool is the standard CVD-safe pairing -- plasma and viridis
# both start in purple, which made the two panels hard to tell apart at a glance. The pale ends
# are trimmed so the lightest points stay visible on white.
# Set CMAP_A, CMAP_B = 'plasma', 'viridis' to go back.
def truncate_cmap(name, lo=0.35, hi=0.95, n=256):
    """A colormap restricted to [lo, hi] of the original's range."""
    base = matplotlib.colormaps[name]
    return LinearSegmentedColormap.from_list(f'{name}_{lo}_{hi}', base(np.linspace(lo, hi, n)))


CMAP_A = truncate_cmap('YlOrBr')      # y1: warm, matches the amber probe / arrow colour
CMAP_B = truncate_cmap('PuBu')        # y2: cool, matches the blue probe / arrow colour
SCAT = dict(s=16, alpha=0.75, edgecolors='none')
FIT_LS, FIT_COL = ':', 'black'

# 3D convention: (x, y, z) = (PC2, PC3, PC1) so PC1 is vertical and increases upward.
PC_ORDER = (1, 2, 0)
PC_NAMES = ('PC1', 'PC2', 'PC3')
VIEW = dict(elev=25, azim=-60, roll=0)      # sweep azim with the angle-picker cell below

# Notation. Both directions are read at the SECOND-number position, so both are v's and the
# subscript says which number each carries: v1 = the first number's direction there (H14's
# DAS direction, written in by the head), v2 = the second number's (L13 PC1, already in
# place). The first number's direction at its OWN position n1 is u, and it belongs to
# makefig_numberreps.ipynb.
V1, V2 = r'$\mathbf{v}_1$', r'$\mathbf{v}_2$'
U = r'$\mathbf{u}$'


def xyz_for_plot(c):
    """Reorder (N, 3) coordinates (or a (3,) direction) so PC1 lands on the vertical axis."""
    return np.asarray(c)[..., list(PC_ORDER)]


def style_axes(ax, xlabel=None, ylabel=None, title=None, grid=True):
    if xlabel: ax.set_xlabel(xlabel, fontsize=LABEL_FS)
    if ylabel: ax.set_ylabel(ylabel, fontsize=LABEL_FS)
    if title:  ax.set_title(title, fontsize=TITLE_FS)
    ax.tick_params(labelsize=TICK_FS)
    if grid: ax.grid(alpha=0.3)
    for side in ('top', 'right'):
        if side in ax.spines: ax.spines[side].set_visible(False)


def aggregate(x, y):
    """Group by number value: (distinct x, mean y, SE). At each value the partner number ranges
    over the whole cloud, so this averages that nuisance variation out. A value seen once gets
    SE 0 rather than a nan."""
    u = np.unique(x)
    m, se = [], []
    for v in u:
        g = y[x == v]
        m.append(g.mean())
        se.append(g.std(ddof=1) / np.sqrt(len(g)) if len(g) > 1 else 0.0)
    return u, np.array(m), np.array(se)


def fit_logx(x, y):
    """Least-squares y = p*log(x) + q; returns (p, q, r2). Same fit data_sharedrep.py saves,
    recomputed here for projections that have no saved log-fit block."""
    p, q = np.polyfit(np.log(x), y, 1)
    pred = p * np.log(x) + q
    return float(p), float(q), 1.0 - float(((y - pred) ** 2).sum()) / float(((y - y.mean()) ** 2).sum())


def style_inset(axi, xlabel, ylabel, label_fs=11, tick_fs=9):
    """Shared styling for the small axes dropped inside a bigger panel: opaque white so the host
    panel's data does not show through, and no top/right spines."""
    axi.set_xlabel(xlabel, fontsize=label_fs)
    axi.set_ylabel(ylabel, fontsize=label_fs)
    axi.tick_params(labelsize=tick_fs)
    axi.set_facecolor('white'); axi.patch.set_alpha(1.0); axi.set_zorder(5)
    for s in ('top', 'right'):
        axi.spines[s].set_visible(False)


def cond_key(name):
    """The name -> array-key transform data_sharedrep.py applies when saving."""
    return name.replace(' ', '_').replace('+', 'and').replace('@', 'at')


def n1_slot_tokens(Dd, case, prefix=''):
    """(token of the number at n1, token of the number at n2) for the sweep prompts.

    `make_batch` defines tok_nc as the number at the **winning** slot, which is n1 only in the
    'n1 larger' case. Scoring "did it name the first slot" straight off tok_nc therefore silently
    inverts the 'n2 larger' case, so which saved array holds the first-slot number has to be
    chosen from the case."""
    nc, b = Dd[f'{prefix}sweep_tok_prompt_nc'], Dd[f'{prefix}sweep_tok_prompt']
    return (nc, b) if case == 'n1 larger' else (b, nc)


def savefig(fig, name):
    if SAVE:
        fig.savefig(f'{FIGS_DIR}/makefig_sharedreps_{name}.png', dpi=DPI, bbox_inches='tight')

In [ ]:
import matplotlib.patheffects as pe
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d.proj3d import proj_transform


class Arrow3D(FancyArrowPatch):
    """A 3D arrow with a real filled head.

    mpl's 3D `quiver` draws its head as two thin line segments, which all but vanish when the
    arrow is near-vertical or seen edge-on. FancyArrowPatch draws a filled polygon sized by
    `mutation_scale`, in points -- absolute, and independent of the arrow's length."""

    def __init__(self, xyz0, xyz1, **kwargs):
        super().__init__((0, 0), (0, 0), **kwargs)
        self._xyz0, self._xyz1 = xyz0, xyz1

    def do_3d_projection(self, renderer=None):
        (x0, y0, z0), (x1, y1, z1) = self._xyz0, self._xyz1
        xs, ys, zs = proj_transform((x0, x1), (y0, y1), (z0, z1), self.axes.M)
        self.set_positions((xs[0], ys[0]), (xs[1], ys[1]))
        return min(zs)

## Figure 1 — H14 DAS component vs the first number

$f(y_1)=p\log y_1+q$ fit on the raw cloud. Inset: the same component against the L13 DAS component
read at the first-number position.

In [ ]:
AGG1 = 'mean'          # 'mean' = one point per distinct y1 with SE; 'raw' = every cloud point
# The component departs from the log law at the bottom of the range, so the all-points fit alone
# understates it; 'both' draws the masked fit (y >= fit_lo) over its own range as well.
FIT1 = 'both'          # 'all' | 'masked' | 'both'
INSET_BOX = [0.56, 0.13, 0.40, 0.38]

comp = D['h14_das_comp'].astype(float)
blk  = MJ['logfit_h14_das_a']
l13c = D['l13_das_comp_n1'].astype(float)

# DAS signs are arbitrary; flip so the component rises with the number it encodes.
sgn = -1.0 if blk['pearson_r'] < 0 else 1.0
comp_s = sgn * comp

fig, ax = plt.subplots(figsize=(7.2, 5.6))
if AGG1 == 'mean':
    xu, mu, se = aggregate(a_vals, comp_s)
    ax.errorbar(xu, mu, yerr=se, fmt='o', ms=5, color=COL['h14'], linestyle='none',
                elinewidth=1.1, capsize=2, alpha=0.9)
else:
    ax.scatter(a_vals, comp_s, color=COL['h14'], **SCAT)
if FIT1 in ('all', 'both'):
    xf = np.linspace(a_vals.min(), a_vals.max(), 300)
    ax.plot(xf, sgn * (blk['p_all'] * np.log(xf) + blk['q_all']), FIT_LS, color=FIT_COL, lw=2.2,
            alpha=0.45 if FIT1 == 'both' else 1.0,
            label=f"all $y_1$:  $R^2$ = {blk['r2_all']:.2f}")
if FIT1 in ('masked', 'both'):
    xm = np.linspace(blk['fit_lo'], a_vals.max(), 300)
    ax.plot(xm, sgn * (blk['p_masked'] * np.log(xm) + blk['q_masked']), FIT_LS, color=FIT_COL,
            lw=2.6, label=f"$y_1 \\geq$ {blk['fit_lo']:.0f}:  $R^2$ = {blk['r2_masked']:.2f}")
style_axes(ax, xlabel='$y_1$', ylabel='H14 DAS component')
ax.legend(fontsize=LEGEND_FS, frameon=True, framealpha=0.92, edgecolor='0.8', loc='upper left')

# Inset: H14 DAS component @n2 against the L13 DAS component @n1, same prompts.
axi = ax.inset_axes(INSET_BOX)
axi.scatter(l13c, comp_s, c=a_vals, cmap=CMAP_A, s=7, alpha=0.65, edgecolors='none')
axi.set_xlabel('L13 DAS comp. @ $n_1$', fontsize=LABEL_FS - 6)
axi.set_ylabel('H14 DAS comp.', fontsize=LABEL_FS - 6)
axi.tick_params(labelsize=TICK_FS - 5)
# r inside the axes, and an opaque background: the main fit line runs through this corner.
axi.text(0.05, 0.08, f"r = {sgn * MJ['h14_das_vs_l13_das_r']:+.2f}", transform=axi.transAxes,
         fontsize=LABEL_FS - 5, ha='left', va='bottom')
axi.set_facecolor('white'); axi.patch.set_alpha(1.0); axi.set_zorder(5)
for s in ('top', 'right'): axi.spines[s].set_visible(False)

print(f"log fit (all points): {blk['p_all']:+.3f} log y {blk['q_all']:+.3f}   "
      f"R2 = {blk['r2_all']:.3f}   pearson r = {blk['pearson_r']:+.3f}")
print(f"masked (y >= {blk['fit_lo']:.0f}): R2 = {blk['r2_masked']:.3f}   n = {blk['n_masked']}")
print(f"vs y2 (control): pearson r = {MJ['logfit_h14_das_b']['pearson_r']:+.3f}")
plt.tight_layout(); savefig(fig, f'{REGIME}_fig1_h14_das_logfit'); plt.show()

## Figure 2 — L14 residual, top-3 PCA

Residual stream leaving layer 14 at the second-number position. Coloured by each number, with the
relevant direction as an arrow from the cloud centre: $\mathbf{v}_1$ (H14 DAS) on the $y_1$ panel,
$\mathbf{v}_2$ (L13 PC1 @ $n_2$) on the $y_2$ panel.

In [ ]:
ARROW_SPAN, ARROW_LW, ARROW_HEAD = 1.00, 3.0, 24
ARROW_LABEL_OFF, ARROW_COL = 1.22, '#111111'

coords = D['pca14_coords']
evr    = D['pca14_evr']
xyz    = xyz_for_plot(coords)

# Sign each arrow so it points toward INCREASING number: a direction's sign is arbitrary, the
# correlation of its component with the number is the ground truth. Components are recomputed from
# the saved cloud so the check is exact for both directions.
resid14 = D['resid14']
comp_of = {'h14_das':  resid14 @ D['h14_das_dir'],
           'l13_pc1b': resid14 @ D['l13_pc1b_dir']}

PANELS = [('h14_das',  a_vals, '$y_1$', CMAP_A, V1,  f'{REGIME}_fig2_l14_pca3_by_y1'),
          ('l13_pc1b', b_vals, '$y_2$', CMAP_B, V2, f'{REGIME}_fig2_l14_pca3_by_y2')]


def pca3_panel(dkey, values, vlab, cmap, arrow_lab, fname):
    fig = plt.figure(figsize=(8.0, 6.8))
    ax = fig.add_subplot(111, projection='3d')

    # Bare axes: no panes, no grid -- the cloud carries the depth cue.
    ax.set_facecolor('none')
    for _axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        _axis.pane.fill = False
        _axis.pane.set_edgecolor('none')
        _axis.line.set_color('0.65')
    ax.grid(False)

    sc = ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], c=values, cmap=cmap, depthshade=False,
                    zorder=3, **SCAT)

    v3 = xyz_for_plot(D[f'{dkey}_in_pca14'])
    if np.corrcoef(values, comp_of[dkey])[0, 1] < 0:
        v3 = -v3
    # Drawn as a pure direction: ||v3|| is the fraction of the unit vector inside the 3D basis
    # (reported as *_pca14_captured), which would otherwise shorten the arrow on top of the scale.
    v3 = v3 / np.linalg.norm(v3)
    centroid = xyz.mean(axis=0)
    scale = ARROW_SPAN * np.ptp(xyz @ v3)
    tail = centroid - v3 * scale * 0.5
    ax.add_artist(Arrow3D(tail, tail + v3 * scale, arrowstyle='-|>', mutation_scale=ARROW_HEAD,
                          lw=ARROW_LW, color=ARROW_COL, shrinkA=0, shrinkB=0, zorder=12))
    _t = ax.text(*(tail + v3 * scale * ARROW_LABEL_OFF), arrow_lab, color=ARROW_COL,
                 fontsize=LABEL_FS + 4)
    _t.set_path_effects([pe.withStroke(linewidth=3, foreground='white')])

    pclab = lambda k: f'{PC_NAMES[k]} ({100 * evr[k]:.0f}%)'
    ax.set_xlabel(pclab(PC_ORDER[0]), fontsize=LABEL_FS - 3)
    ax.set_ylabel(pclab(PC_ORDER[1]), fontsize=LABEL_FS - 3)
    ax.set_zlabel(pclab(PC_ORDER[2]), fontsize=LABEL_FS - 3)
    for _ax in (ax.xaxis, ax.yaxis, ax.zaxis): _ax.labelpad = 12
    ax.tick_params(labelsize=TICK_FS - 3, pad=1)
    ax.view_init(**VIEW)
    cb = fig.colorbar(sc, ax=ax, shrink=0.58, pad=0.14)
    cb.set_label(vlab, fontsize=LABEL_FS + 2); cb.ax.tick_params(labelsize=TICK_FS)
    cb.outline.set_visible(False)
    print(f"{arrow_lab} in the 3D basis: {np.round(D[f'{dkey}_in_pca14'], 3).tolist()}   "
          f"norm captured {float(D[f'{dkey}_pca14_captured']):.3f}")
    fig.subplots_adjust(left=0.02, right=0.94, top=0.97, bottom=0.04)
    savefig(fig, fname); plt.show()


print(f'L14 PCA EVR: {np.round(evr, 3).tolist()}')
for args in PANELS:
    pca3_panel(*args)

In [ ]:
# Angle picker: the same cloud from a grid of viewing angles. Read off the pair you want, set VIEW
# in the style cell, and re-run the cell above. No recomputation -- just re-rendering coords.
ELEVS, AZIMS = [10, 25, 40], [-90, -60, -30, 0]

fig = plt.figure(figsize=(3.0 * len(AZIMS), 2.9 * len(ELEVS)))
for r, e in enumerate(ELEVS):
    for c, az in enumerate(AZIMS):
        ax = fig.add_subplot(len(ELEVS), len(AZIMS), r * len(AZIMS) + c + 1, projection='3d')
        ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], c=a_vals, cmap=CMAP_A, s=3, alpha=0.5,
                   edgecolors='none')
        ax.view_init(elev=e, azim=az)
        ax.set_title(f'elev={e}, azim={az}', fontsize=9)
        for axis in (ax.xaxis, ax.yaxis, ax.zaxis): axis.set_ticklabels([])
# tight_layout does not handle a grid of 3D axes; set the margins directly.
fig.subplots_adjust(left=0.02, right=0.98, top=0.94, bottom=0.02, wspace=0.05, hspace=0.12)
plt.show()

## Figure 3 — per-head patching across layer 14

Each head's own output contribution interchanged at the second-number position, alone and jointly
with an L13 interchange at that same position.

In [ ]:
# Which joint arm to show alongside 'alone'. 'with_L13_PC1_n2' is the rank-1 control,
# 'with_L13_full_n2' the full-rank one; both are in results/.
ARMS = ['alone', 'with_L13_PC1_n2']
ARM_LABEL = {'alone': 'head alone',
             'with_L13_PC1_n2': r'+ L13 $\mathbf{v}_2$ @ $n_2$',
             'with_L13_full_n2': '+ L13 full @ $n_2$'}
ARM_COL = {'alone': COL['full'], 'with_L13_PC1_n2': COL['das'], 'with_L13_full_n2': COL['pc1']}
# Every joint bar carries the L13 patch's own effect; the reference line is that alone value, so a
# per-head bar is read as what the head adds on top of it.
ARM_REF = {'with_L13_PC1_n2': 'L13_PC1_atn2', 'with_L13_full_n2': 'L13_full_atn2'}

# Heads are ordered by effect, not by index: with 28 heads and one that matters, index order
# buries the result in a flat row. Both panels share the order so a column is one head.
SORT_ARM, SORT_METRIC = ARMS[-1], 'posrec'      # rank by the last arm shown

H_HEAD, N_HEADS = M['h_head'], M['n_heads']
order = np.argsort(-D[f'head_{SORT_METRIC}_{SORT_ARM}'].mean(axis=1))
heads = np.arange(N_HEADS)
W = 0.8 / len(ARMS)

fig, axes = plt.subplots(2, 1, figsize=(15, 8.4), sharex=True)
for ax, (metric, ylab) in zip(axes, [('posrec', 'position recovery'), ('iia', 'IIA')]):
    for ai, arm in enumerate(ARMS):
        v = D[f'head_{metric}_{arm}'][order]                # (n_heads, n_eval), ranked
        mu = v.mean(axis=1)
        se = v.std(axis=1) / np.sqrt(v.shape[1])
        ax.bar(heads + (ai - (len(ARMS) - 1) / 2) * W, mu, W, yerr=se, capsize=2,
               color=ARM_COL[arm], alpha=0.9, label=ARM_LABEL[arm])
    for arm in ARMS:
        if arm in ARM_REF:
            ref = D[f'cond_{metric}_{ARM_REF[arm]}'].mean()
            ax.axhline(ref, ls='--', color=ARM_COL[arm], lw=1.3, alpha=0.8, zorder=1,
                       label=f'{ARM_LABEL[arm]}, L13 alone')
    if metric == 'iia':
        ax.set_ylim(0, 1.05)
    ax.axhline(0, color='0.6', lw=0.9)
    ax.axvspan(*(np.where(order == H_HEAD)[0][0] + np.array([-0.5, 0.5])),
               color='0.85', alpha=0.45, zorder=0)                               # the copy head
    style_axes(ax, ylabel=ylab)
axes[0].legend(fontsize=LEGEND_FS, ncol=len(ARMS), frameon=True, framealpha=0.92, edgecolor='0.8')
axes[-1].set_xticks(heads)
axes[-1].set_xticklabels([f'H{h}' for h in order], fontsize=TICK_FS - 3, rotation=90)
style_axes(axes[-1], xlabel=f"layer {M['h_layer']} head (ranked by {SORT_ARM} {SORT_METRIC})")

for arm in ARMS:
    m = D[f'head_iia_{arm}'].mean(axis=1)
    top = np.argsort(-m)[:5]
    print(f"{arm:>20}: top IIA  " + '  '.join(f'H{h}={m[h]:.3f}' for h in top))
plt.tight_layout(); savefig(fig, f'{REGIME}_fig3_head_sweep'); plt.show()

## Figure 4 — direction alignment

Cosine similarity between the H14 DAS direction, L13 PC1 at $n_2$, and the two probe directions
(ridge probes for $\log y_1$ and $\log y_2$ from the L14 residual). Chance for two random unit
vectors is $1/\sqrt{d}$.

In [ ]:
SHOW_H14_PC1 = False      # the saved matrix carries H14 PC1 as a 5th row; True includes it

names_all = M['cos_names']
keep = (list(range(len(names_all))) if SHOW_H14_PC1
        else [i for i, n in enumerate(names_all) if n != 'H14 PC1'])
names = [names_all[i] for i in keep]
Cm = D['cos_matrix'][np.ix_(keep, keep)]
d_model = M['d_model']

fig, ax = plt.subplots(figsize=(6.8, 5.8))
im = ax.imshow(Cm, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=35, ha='right', fontsize=TICK_FS)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=TICK_FS)
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f'{Cm[i, j]:+.2f}', ha='center', va='center', fontsize=TICK_FS,
                color='white' if abs(Cm[i, j]) > 0.6 else 'black')
cb = fig.colorbar(im, ax=ax, shrink=0.85); cb.set_label('cosine similarity', fontsize=LABEL_FS - 2)
ax.set_title(f'chance $\\approx$ {1 / np.sqrt(d_model):.3f}', fontsize=TITLE_FS - 2)
ax.tick_params(length=0)
for s in ax.spines.values(): s.set_visible(False)

print(f"probe log A: alpha={MJ['probe_logA_stats']['alpha']:.3g}  "
      f"held-out R2={MJ['probe_logA_stats']['r2_test']:.3f}")
print(f"probe log B: alpha={MJ['probe_logB_stats']['alpha']:.3g}  "
      f"held-out R2={MJ['probe_logB_stats']['r2_test']:.3f}")
plt.tight_layout(); savefig(fig, f'{REGIME}_fig4_direction_cosines'); plt.show()

## Figure 5 — interchange interventions at the second-number position

With $\Delta(\ell)=\ell_{t_{\mathrm{nc}}}-\ell_{t_{b}}$ the last-position logit difference between
the counterfactual answer token $t_{\mathrm{nc}}$ and the clean answer token $t_{b}$,

$$
\mathrm{PR}\;=\;\frac{\Delta(\ell_{\text{patched}})-\Delta(\ell_{\text{corr}})}
                     {\Delta(\ell_{\text{clean}})-\Delta(\ell_{\text{corr}})},
\qquad
\mathrm{IIA}\;=\;\mathbb{1}\!\left[\arg\max_{t}\;\ell_{\text{patched},\,t}=t_{\mathrm{nc}}\right].
$$

Hatched bar = joint scope. `L14 full @ n2` is the ceiling: the whole residual stream leaving
layer 14 replaced.

In [ ]:
SHOW_POSREC = True        # second panel with position recovery alongside IIA

COND_COL = {'L13 full @n2': COL['full'], 'L13 PC1 @n2': COL['pc1'],
            'H14 DAS': COL['h14'], 'H14 DAS + L13 PC1 @n2': COL['h14'],
            # every L14-site condition is grey: same site, the bars differ only in rank.
            'L14 u @n2': COL['ceil'], 'L14 v2 @n2': COL['ceil'],
            'L14 uv plane @n2': COL['ceil'], 'L14 full @n2': COL['ceil'],
            'L14 uv plane preMLP @n2': COL['ceil'], 'L14 full preMLP @n2': COL['ceil']}
COND_SHORT = {'L13 full @n2': 'L13 full\n@ $n_2$',
              'L13 PC1 @n2': r'L13 $\mathbf{v}_2$' + '\n@ $n_2$',
              'H14 DAS': r'H14 $\mathbf{v}_1$',
              'H14 DAS + L13 PC1 @n2': r'H14 $\mathbf{v}_1$ +' + '\n' + r'L13 $\mathbf{v}_2$ @ $n_2$',
              'L14 u @n2': f'L14 {V1}\n@ $n_2$',
              'L14 v2 @n2': f'L14 {V2}\n@ $n_2$',
              'L14 uv plane @n2': f'L14 span({V1},{V2})' + '\n@ $n_2$',
              'L14 full @n2': 'L14 full\n@ $n_2$',
              'L14 uv plane preMLP @n2': f'L14 span({V1},{V2})' + '\npre-MLP',
              'L14 full preMLP @n2': 'L14 full\npre-MLP'}
# Rank-restricted scopes are hatched, so a hatched bar always means "a subspace, not the whole
# thing" -- the joint H14+L13 condition and the rank-2 plane both qualify.
HATCHED = {'H14 DAS + L13 PC1 @n2', 'L14 uv plane @n2', 'L14 uv plane preMLP @n2'}

names = list(M['condition_names'])
metrics = ([('iia', 'IIA', (0, 1.08))] +
           ([('posrec', 'position recovery', None)] if SHOW_POSREC else []))

fig, axes = plt.subplots(1, len(metrics), figsize=(1.55 * len(names) * len(metrics), 5.6),
                         squeeze=False)
for ax, (metric, ylab, ylim) in zip(axes[0], metrics):
    for i, nm in enumerate(names):
        v = D[f'cond_{metric}_{cond_key(nm)}']
        se = v.std() / np.sqrt(len(v))
        ax.bar(i, v.mean(), 0.66, yerr=se, capsize=4, color=COND_COL[nm], alpha=0.9,
               hatch=JOINT_HATCH if nm in HATCHED else '', edgecolor='white', linewidth=0)
        ax.text(i, v.mean() + se + 0.02 * np.ptp(ax.get_ylim()), f'{v.mean():.2f}',
                ha='center', va='bottom', fontsize=TICK_FS)
    if metric == 'posrec':
        ax.axhline(1.0, ls='--', color='0.6', lw=0.9)
    if ylim: ax.set_ylim(*ylim)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels([COND_SHORT[n] for n in names], fontsize=TICK_FS - 2)
    style_axes(ax, ylabel=ylab)

for nm in names:
    r, i = D[f'cond_posrec_{cond_key(nm)}'], D[f'cond_iia_{cond_key(nm)}']
    print(f'{nm:>24}: position recovery {r.mean():6.3f} +/- {r.std() / np.sqrt(len(r)):.3f}   '
          f'IIA {i.mean():.3f}')
plt.tight_layout(); savefig(fig, f'{REGIME}_fig5_conditions'); plt.show()

## Combined paper figure — the joint representation

Everything here is a claim about the **pair** of directions, not about either one. Provenance and
each direction's own causal role live in `makefig_numberreps.ipynb`; this figure starts from them.

(a)–(c) the L14 pre-MLP cloud in span($\mathbf{v}_1$, $\mathbf{v}_2$), by $y_1$, by $y_2$, and by
which is larger · (d) pairwise cosines of $\mathbf{u}$, $\mathbf{v}_1$, $\mathbf{v}_2$, band $\pm 2/\sqrt{d}$ ·
(e) IIA increments: neither direction alone suffices here, the pair reaches the full-rank ceiling ·
(f) the dose-response sweep of the plane.

Axes: $\mathbf{v}_2^{\perp}\propto\mathbf{v}_2-(\mathbf{v}_2\cdot\mathbf{v}_1)\,\mathbf{v}_1$.

In [ ]:
# ==================================================================================
# FONT SIZES  --  edit any of these
# ==================================================================================
FS_AXIS        = 26   # x / y axis labels, every panel
FS_TICK        = 20   # numeric tick labels, every panel
FS_PANEL       = 31   # the bold (a) ... (f)
FS_CBAR_TICK   = 18   # colorbar tick labels, every panel

FS_CBAR_LABEL  = 25   # (a)-(c)  y1 / y2 / "y1 > y2" beside the colorbars
FS_COMPASS     = 21   # (a)-(c)  v1 / v2 beside the compass arrows
FS_ANGLE       = 18   # (a)      the angle printed over the compass

FS_COS_TICK    = 24   # (d)      pair names under the bars

FS_LEGEND      = 20   # (e)      legend entries
FS_BAR_TICK    = 22   # (e)      component names under the bars

FS_CONTOUR     = 17   # (f)      the numbers on the contour lines
FS_SWEEP_CBAR  = 22   # (f)      colorbar label

# ==================================================================================
# LAYOUT  --  edit any of these
# ==================================================================================
FIG_SIZE    = (23.0, 12.2)   # whole figure, inches
GRID_WSPACE = 0.30           # horizontal gap between panels
GRID_HSPACE = 0.30           # vertical gap between rows
CLOUD_SIZE  = 11             # marker size for the cloud in (a)-(c)
SWEEP_CLOUD_SIZE = 6         # marker size for the grey cloud in (f)
CONTOUR_LW  = 2.4            # contour line width in (f); the 0.5 boundary is drawn heavier
BOUNDARY_LW = 3.0            # the P = 0.5 boundary in (f)


UV_SITE_FIG = 'pre'        # the plane is drawn where the rank-2 patch works; 'post' for contrast
COMPASS_BOX = [0.72, 0.70, 0.27, 0.28]   # upper right: empty in every panel, (f) included
PANEL_LX, PANEL_LY = -0.15, 1.02

_cloud_key = {'pre': 'resid14_pre', 'post': 'resid14'}[UV_SITE_FIG]
if _cloud_key not in D:
    print(f"'{_cloud_key}' missing -- falling back to the end-of-layer cloud")
    _cloud_key = 'resid14'

fig = plt.figure(figsize=FIG_SIZE)
gs = fig.add_gridspec(2, 3, wspace=GRID_WSPACE, hspace=GRID_HSPACE)


def panel_letter(ax, s):
    ax.text(PANEL_LX, PANEL_LY, s, transform=ax.transAxes, fontsize=FS_PANEL,
            fontweight='bold', ha='left', va='bottom')


# ===================================== (a)-(c) the cloud inside span(v1, v2) ====================
# Gram-Schmidt, not a PCA of the pair: e1 is exactly v1, so the horizontal axis IS the component
# the H14 patch writes, and since cos(v1, v2) = -0.08 the vertical axis is 99.7% v2.
_v1 = D['h14_das_dir'].astype(float);  _v1 /= np.linalg.norm(_v1)
_v2 = D['l13_pc1b_dir'].astype(float); _v2 /= np.linalg.norm(_v2)
_R = D[_cloud_key].astype(float)
_Xc = _R - _R.mean(axis=0)
if np.corrcoef(a_vals, _Xc @ _v1)[0, 1] < 0: _v1 = -_v1
if np.corrcoef(b_vals, _Xc @ _v2)[0, 1] < 0: _v2 = -_v2
_e2 = _v2 - (_v2 @ _v1) * _v1; _e2 /= np.linalg.norm(_e2)
cx, cy = _Xc @ _v1, _Xc @ _e2
cos_uv = float(_v1 @ _v2)
ang_uv = np.degrees(np.arccos(cos_uv))
# v1 and v2 are not orthogonal, so the vertical axis is v2 with its v1 part removed: v2-perp.
XLAB = r'$f_1(y_1)$, comp. along $\mathbf{v}_1$'
YLAB = r'comp. along $\mathbf{v}_2^{\perp}$'


def compass(ax, show_angle=False, box=None):
    """v1 and v2 at their true angle, in an equal-aspect inset (the panel's own aspect is set
    by the data, so arrows drawn in data coordinates would show the wrong angle)."""
    axi = ax.inset_axes(box or COMPASS_BOX); axi.set_aspect('equal')
    for vec, nm in (((1.0, 0.0), V1), ((cos_uv, np.sqrt(1 - cos_uv ** 2)), V2)):
        axi.annotate('', xy=vec, xytext=(0, 0),
                     arrowprops=dict(arrowstyle='-|>', lw=2.3, color='#111111', mutation_scale=15))
        axi.text(1.34 * vec[0], 1.34 * vec[1], nm, fontsize=FS_COMPASS, fontweight='bold',
                 ha='center', va='center')
    axi.set_xlim(-0.5, 1.6); axi.set_ylim(-0.35, 1.7); axi.axis('off')
    if show_angle:
        axi.set_title(f'$\\angle$ = {ang_uv:.1f}$^\\circ$', fontsize=FS_ANGLE, pad=0)

# (c): the PuOr ends (f) uses -- orange = y1 is the larger, purple = y2 is.
ORDER_CMAP = matplotlib.colors.ListedColormap(['#998EC3', '#F1A340'])
PLANE = [('(a)', a_vals, '$y_1$', CMAP_A, None),
         ('(b)', b_vals, '$y_2$', CMAP_B, None),
         ('(c)', (a_vals > b_vals).astype(float), '$y_1 > y_2$', ORDER_CMAP, (0, 1))]

for k, (plet, vals, vlab, cmap, clim) in enumerate(PLANE):
    ax = fig.add_subplot(gs[0, k])
    sc = ax.scatter(cx, cy, c=vals, cmap=cmap, s=CLOUD_SIZE, alpha=0.72, edgecolors='none',
                    **({} if clim is None else dict(vmin=clim[0], vmax=clim[1])))
    ax.set_xlabel(XLAB, fontsize=FS_AXIS)
    ax.set_ylabel(YLAB, fontsize=FS_AXIS)
    ax.tick_params(labelsize=FS_TICK)
    for s_ in ('top', 'right'):
        ax.spines[s_].set_visible(False)
    cb = fig.colorbar(sc, ax=ax, shrink=0.85, pad=0.02)
    cb.set_label(vlab, fontsize=FS_CBAR_LABEL); cb.ax.tick_params(labelsize=FS_CBAR_TICK)
    cb.outline.set_visible(False)
    if clim is not None:
        cb.set_ticks([0.25, 0.75]); cb.set_ticklabels(['no', 'yes'])
    compass(ax, show_angle=(k == 0))   # the angle is one measurement; annotate it once
    panel_letter(ax, plet)

# ===================================== (d) direction cosines ====================================
# Signed cosines between u, v1, v2, each signed to rise with its own number, so the sign means
# something. u is read at n1 and the v's at n2, but all three live in d_model. Grey band: +-2 sd of
# the cosine between two random unit vectors in d_model (sd = 1/sqrt(d)). Whiskers on the u bars:
# the range over u's DAS seeds (numberreps; seed 0 is u itself). v1 has a single seed.
ax = fig.add_subplot(gs[1, 0])
_u = D['l13_das_dir'].astype(float); _u /= np.linalg.norm(_u)
if np.corrcoef(a_vals, D['l13_das_comp_n1'])[0, 1] < 0: _u = -_u
_useeds = np.load(f'{RESULTS_DIR}/numberreps_{REGIME}.npz')['das_seed_dirs'].astype(float)
_useeds /= np.linalg.norm(_useeds, axis=1, keepdims=True)
_useeds *= np.sign(_useeds @ _u)[:, None]                 # each seed signed like u
COS_PAIRS = [(f'{U},{V1}', _u @ _v1, _useeds @ _v1),
             (f'{U},{V2}', _u @ _v2, _useeds @ _v2),
             (f'{V1},{V2}', _v1 @ _v2, None)]
_chance = 1 / np.sqrt(M['d_model'])
ax.axhspan(-2 * _chance, 2 * _chance, color='0.85', lw=0, zorder=0, label=r'random pairs ($\pm 2\sigma$)')
ax.axhline(0, color='0.3', lw=1.0, zorder=1)
for k, (lab, val, seeds) in enumerate(COS_PAIRS):
    yerr = (None if seeds is None else                    # clip: float32 seeds vs float64 u
            [[max(val - seeds.min(), 0)], [max(seeds.max() - val, 0)]])
    ax.bar(k, val, 0.6, yerr=yerr, capsize=5, color=COL['h14'], alpha=0.9, linewidth=0, zorder=2)
ax.set_xticks(range(len(COS_PAIRS)))
ax.set_xticklabels([c_[0] for c_ in COS_PAIRS], fontsize=FS_COS_TICK)
ax.set_xlim(-0.6, len(COS_PAIRS) - 0.4)
ax.set_ylim(-0.2, 0.6)
style_axes(ax, ylabel='cosine similarity', grid=False)
ax.set_ylabel('cosine similarity', fontsize=FS_AXIS); ax.tick_params(labelsize=FS_TICK)
ax.legend(fontsize=FS_LEGEND, frameon=False, loc='upper right')
print('  (d) ' + '   '.join(f"cos({l_.replace('$', '').replace(chr(92) + 'mathbf', '')}) {v_:+.3f}"
                           + ('' if s_ is None else f' [seeds {s_.min():+.3f}, {s_.max():+.3f}]')
                           for l_, v_, s_ in COS_PAIRS)
      + f'   random-pair sd = 1/sqrt(d) = {_chance:.3f}')
panel_letter(ax, '(d)')

# ===================================== (e) IIA increments =======================================
# Not "these directions are causal" -- that is the other figure's panel. The claim here is the
# INCREMENT: neither half does anything at this position on its own, and the rank-2 plane reaches
# the full-rank ceiling. Colour marks the role: green = control, purple = the span(v1, v2)
# hypothesis, grey = ceiling. Purple and green are the ColorBrewer PRGn endpoints, a published
# colourblind-safe pair, and neither hue appears elsewhere in this figure.
ax = fig.add_subplot(gs[1, 1])
# Colour = which number the counterfactual perturbs, warm for y1 and cool for y2, the same
# convention panels (a) and (b) use for the two numbers. Hatching still marks a rank-restricted
# scope. Naming the arms by what is perturbed rather than by "n1/n2 larger" is what makes the
# dissociation readable: each direction moves the answer exactly when its own number changed.
E_ARMS = [('', f'$y_1$ perturbed', COL['das']),
          ('alt_', f'$y_2$ perturbed', COL['full'])]
E_ARMS = [a for a in E_ARMS if f'{a[0]}cond_iia_{cond_key("H14 DAS")}' in D]
COND_E = [('L13 full @n2',            r'$\mathbf{z}^2_{l=13}$', False),
          ('L13 PC1 @n2',             V2,                   False),
          ('H14 DAS',                 V1,                   False),
          ('H14 DAS + L13 PC1 @n2',   f'{V1}&{V2}',         True),
          ('L14 uv plane preMLP @n2', f'{V1},{V2}\nplane',  True),
          ('L14 full @n2',            r'$\mathbf{z}^2_{l=14}$', False)]
COND_E = [c for c in COND_E if f'cond_iia_{cond_key(c[0])}' in D]
xe, We = np.arange(len(COND_E)), 0.8 / max(len(E_ARMS), 1)
for ai, (pre, lab, col) in enumerate(E_ARMS):
    xs = xe + (ai - (len(E_ARMS) - 1) / 2) * We
    for k, (nm, _lab, hatch) in enumerate(COND_E):
        v = D.get(f'{pre}cond_iia_{cond_key(nm)}')
        if v is None:
            continue
        se = v.std() / np.sqrt(len(v))
        ax.bar(xs[k], v.mean(), We, yerr=se, capsize=3, color=col, alpha=0.92, linewidth=0,
               label=lab if k == 0 else None)
ax.set_ylim(0, 1.22)        # headroom: the legend sits over the tallest bar otherwise
ax.set_xticks(xe)
ax.set_xticklabels([c[1] for c in COND_E], fontsize=FS_BAR_TICK)
ax.legend(fontsize=FS_LEGEND, frameon=False, loc='upper left', ncol=2, handlelength=1.2,
          columnspacing=1.0)
style_axes(ax, ylabel='IIA', grid=False)
ax.set_xlabel('patched component', fontsize=FS_AXIS, labelpad=10)
ax.set_ylabel('IIA', fontsize=FS_AXIS); ax.tick_params(labelsize=FS_TICK)
for nm, _l, _h in COND_E:
    print('  (e) ' + f'{nm:>26}   ' + '   '.join(
        f'{lab}: {D[f"{pre}cond_iia_{cond_key(nm)}"].mean():.3f}'
        for pre, lab, _c in E_ARMS if f'{pre}cond_iia_{cond_key(nm)}' in D))
panel_letter(ax, '(e)')

# ===================================== (f) the plane as a dose-response surface =================
# The coordinates are SET to the values a chosen pair of numbers would put there, over a grid.
# The model only names numbers that are in its prompt, so what the implanted pair can decide is
# which SLOT wins -- which is the mechanism's own claim.
ax = fig.add_subplot(gs[1, 2])
if 'sweep_argmax' not in D or 'uv_plane_basis' not in D:
    ax.set_axis_off()
    print('  (f) sweep not in this npz -- re-run data_sharedrep.py')
else:
    sv, sam = D['sweep_values'], D['sweep_argmax']
    G = len(sv)
    _tn1, _tn2 = n1_slot_tokens(D, M['case'])
    _t1, _t2 = _tn1[None, :], _tn2[None, :]
    P1 = np.full((G, G), np.nan)                       # rows y1*, cols y2*
    for i in range(G):
        for j in range(G):
            if i != j:
                n1, n2 = (sam[i, j] == _t1).sum(), (sam[i, j] == _t2).sum()
                P1[i, j] = n1 / max(n1 + n2, 1)

    # Put the grid on the SAME axes as (a)-(c) so the sweep and the cloud can be read together.
    # sweep_coords are in the raw uv_plane_basis the patch used; (cx, cy) are centred and
    # sign-oriented, so map one into the other exactly rather than re-estimating.
    _Puv = D['uv_plane_basis'].astype(float)
    _raw_mu = (_R @ _Puv).mean(axis=0)
    _sgn = np.array([np.sign(_v1 @ _Puv[:, 0]) or 1.0, np.sign(_e2 @ _Puv[:, 1]) or 1.0])
    _g = (D['sweep_coords'].astype(float) - _raw_mu) * _sgn      # (G, 2) in the panel's frame
    gx, gy = _g[:, 0], _g[:, 1]

    ax.scatter(cx, cy, s=SWEEP_CLOUD_SIZE, color='0.72', alpha=0.45, edgecolors='none', zorder=0)
    # Contours cannot cross the missing diagonal, so those cells are filled from their two in-row
    # neighbours for the contour only.
    Pc = P1.copy()
    for k in range(G):
        Pc[k, k] = np.mean([Pc[k, q] for q in (k - 1, k + 1) if 0 <= q < G])
    # Diverging about 0.5 with the figure's own semantics: orange (warm) = names the first
    # number, purple (cool) = names the second, matching the y1/y2 ramps in (a) and (b). PuOr is
    # a published colourblind-safe diverging map. Its midpoint is white, so the 0.5 level -- the
    # decision boundary, and the result -- is drawn separately in black.
    LEVELS = [0.1, 0.25, 0.75, 0.9]
    _norm = matplotlib.colors.Normalize(0, 1)
    cs = ax.contour(gx, gy, Pc.T, levels=LEVELS, cmap='PuOr_r', norm=_norm,   # .T: x is y1* now
                    linewidths=CONTOUR_LW, zorder=4)
    ax.clabel(cs, inline=True, fontsize=FS_CONTOUR, fmt='%.2f')
    cs5 = ax.contour(gx, gy, Pc.T, levels=[0.5], colors='#111111', linewidths=BOUNDARY_LW,
                     zorder=5)
    ax.clabel(cs5, inline=True, fontsize=FS_CONTOUR, fmt='%.2f')
    _sm = matplotlib.cm.ScalarMappable(norm=_norm, cmap='PuOr_r'); _sm.set_array([])
    cbf = fig.colorbar(_sm, ax=ax, shrink=0.85, pad=0.02, ticks=[0, 0.25, 0.5, 0.75, 1])
    cbf.set_label(r'P(model answer $= y_1$)', fontsize=FS_SWEEP_CBAR)
    cbf.ax.tick_params(labelsize=FS_CBAR_TICK); cbf.outline.set_visible(False)
    cbf.ax.axhline(0.5, color='#111111', lw=BOUNDARY_LW)        # the black boundary's level
    ax.set_xlabel(XLAB, fontsize=FS_AXIS)
    ax.set_ylabel(YLAB, fontsize=FS_AXIS)
    ax.tick_params(labelsize=FS_TICK)
    compass(ax, box=[0.74, 0.74, 0.24, 0.25])       # smaller and higher: clear of the 0.75 contour
    for s_ in ('top', 'right'):
        ax.spines[s_].set_visible(False)
    _off = ~np.eye(G, dtype=bool)
    _up = np.triu(np.ones((G, G), bool), 1).T & _off
    print(f'  (f) P(first slot) {np.nanmean(P1[_up]):.3f} when y1* > y2*, '
          f'{np.nanmean(P1[_off & ~_up]):.3f} when y1* < y2*; sign matches on '
          f'{np.mean((P1[_off] > 0.5) == (sv[:, None] > sv[None, :])[_off]):.3f} of cells')
panel_letter(ax, '(f)')

print(f'  plane: {_cloud_key}, cos(v1, v2) = {cos_uv:+.3f} -> {ang_uv:.1f} deg')
for nm, c, vv in (('v1', cx, a_vals), ('v2', cy, b_vals)):
    pnm = np.array([c[vv == q].mean() for q in vv])
    print(f'    {nm}: per-number-mean R2 {1 - ((c - pnm) ** 2).sum() / ((c - c.mean()) ** 2).sum():.3f}')

fig.subplots_adjust(left=0.05, right=0.985, top=0.96, bottom=0.07)
savefig(fig, f'{REGIME}_paperfig')
# plt.savefig('./figs_mainpaper/sharedrep_paperfig.pdf', bbox_inches='tight', dpi=300)
plt.show()

## Figure 6 — the L14 cloud inside span($\mathbf{v}_1$, $\mathbf{v}_2$)

Orthonormal basis $\mathbf{e}_1=\mathbf{v}_1$, $\mathbf{e}_2\propto\mathbf{v}_2-(\mathbf{v}_2\!\cdot\!\mathbf{v}_1)\,\mathbf{v}_1$;
the L14 residual at $n_2$ projected on it, coloured by each number and by which number is larger.
Inset compass: the two directions at their true angle.

In [ ]:
COMPASS_BOX = [0.03, 0.05, 0.24, 0.26]     # lower-left corner of each panel; the cloud misses it
SHOW_ORDER_PANEL = True                    # third panel: which of the two numbers is larger
# Which read-out point. 'pre' is the default now that both are measured: at the pre-MLP site the
# u coordinate is still y1's magnitude (a per-number mean explains R2 = 0.87 of it), while by the
# end of layer 14 the MLP has turned it into the comparison outcome (R2 = 0.35, and its sign
# agrees with y1 > y2 on 96% of the cloud). 'post' redraws the end-of-layer version; the printout
# under the figure reports both statistics either way.
UV_SITE = 'pre'                            # 'pre' | 'post'
UV_CLOUD = {'pre': 'resid14_pre', 'post': 'resid14'}[UV_SITE]
if UV_CLOUD not in D:
    print(f"'{UV_CLOUD}' not in this npz -- falling back to the end-of-layer cloud")
    UV_CLOUD = 'resid14'

# The plane itself. Gram-Schmidt rather than a PCA of the two: e1 is exactly u, so the horizontal
# axis is literally "the component the H14 patch writes", and e2 absorbs only the -0.08 of v2 that
# is not orthogonal to it.
u_dir = D['h14_das_dir'].astype(float);  u_dir /= np.linalg.norm(u_dir)
v_dir = D['l13_pc1b_dir'].astype(float); v_dir /= np.linalg.norm(v_dir)
_R = D[UV_CLOUD].astype(float)
Xc = _R - _R.mean(axis=0)
# Signs are arbitrary; orient each so its component rises with the number it carries.
if np.corrcoef(a_vals, Xc @ u_dir)[0, 1] < 0: u_dir = -u_dir
if np.corrcoef(b_vals, Xc @ v_dir)[0, 1] < 0: v_dir = -v_dir
e1 = u_dir
e2 = v_dir - (v_dir @ u_dir) * u_dir; e2 /= np.linalg.norm(e2)
cx, cy = Xc @ e1, Xc @ e2
cos_uv = float(u_dir @ v_dir)
ang_uv = np.degrees(np.arccos(cos_uv))

PANELS_UV = [(a_vals, '$y_1$', CMAP_A, None), (b_vals, '$y_2$', CMAP_B, None)]
if SHOW_ORDER_PANEL:
    PANELS_UV.append(((a_vals > b_vals).astype(float), '$y_1 > y_2$',
                      matplotlib.colors.ListedColormap([COL['pc1'], COL['h14']]), (0, 1)))

fig, axes = plt.subplots(1, len(PANELS_UV), figsize=(6.4 * len(PANELS_UV), 5.8))
for ax, (vals, vlab, cmap, clim) in zip(np.atleast_1d(axes), PANELS_UV):
    sc = ax.scatter(cx, cy, c=vals, cmap=cmap, s=13, alpha=0.72, edgecolors='none',
                    **({} if clim is None else dict(vmin=clim[0], vmax=clim[1])))
    ax.axhline(0, color='0.78', lw=0.8, zorder=0); ax.axvline(0, color='0.78', lw=0.8, zorder=0)
    style_axes(ax, xlabel=f'coordinate along {V1}', ylabel=f'coordinate along {V2}')
    cb = fig.colorbar(sc, ax=ax, shrink=0.85, pad=0.02)
    cb.set_label(vlab, fontsize=LABEL_FS); cb.ax.tick_params(labelsize=TICK_FS)
    cb.outline.set_visible(False)
    if clim is not None:
        cb.set_ticks([0.25, 0.75]); cb.set_ticklabels(['no', 'yes'])

    # Compass: the panel's aspect ratio is set by the data, so the two directions are drawn in
    # their own equal-aspect axes -- that is the only place the 95 deg is geometrically true.
    axi = ax.inset_axes(COMPASS_BOX); axi.set_aspect('equal')
    for vec, nm in (((1.0, 0.0), V1), ((cos_uv, np.sqrt(1 - cos_uv ** 2)), V2)):
        axi.annotate('', xy=vec, xytext=(0, 0),
                     arrowprops=dict(arrowstyle='-|>', lw=2.4, color='#111111', mutation_scale=16))
        axi.text(1.32 * vec[0], 1.32 * vec[1], nm, fontsize=LABEL_FS - 3, fontweight='bold',
                 ha='center', va='center')
    axi.set_xlim(-0.5, 1.6); axi.set_ylim(-0.35, 1.65); axi.axis('off')
    axi.set_title(f'$\\angle$ = {ang_uv:.1f}$^\\circ$', fontsize=TICK_FS - 2, pad=0)

print(f"{UV_CLOUD}:  cos(u, v2) = {cos_uv:+.3f}  ->  {ang_uv:.1f} deg")
print(f'variance of the L14 cloud inside the plane: '
      f'{(cx.var() + cy.var()) / Xc.var(axis=0).sum():.3f} of the total')
# Per-number-mean R2 says whether an axis is carrying that number's *magnitude*; the sign
# agreement says whether it is instead carrying the comparison outcome. They come apart between
# the two sites, which is the whole reason UV_SITE exists.
for nm, c, vals in (('u', cx, a_vals), ('v2 (perp)', cy, b_vals)):
    pnm = np.array([c[vals == v].mean() for v in vals])
    print(f'  {nm:>10}: corr with log y1 {np.corrcoef(c, np.log(a_vals))[0, 1]:+.3f}   '
          f'with log y2 {np.corrcoef(c, np.log(b_vals))[0, 1]:+.3f}   '
          f'per-number-mean R2 {1 - ((c - pnm) ** 2).sum() / ((c - c.mean()) ** 2).sum():.3f}   '
          f'sign agrees with y1>y2 on {((c > 0) == (a_vals > b_vals)).mean():.3f}')
plt.tight_layout(); savefig(fig, f'{REGIME}_fig6_uv_plane_{UV_SITE}'); plt.show()

## Figure 7 — patching layer 14 inside span($\mathbf{v}_1$, $\mathbf{v}_2$)

Rank-2 interchange of the L14 residual at $n_2$ restricted to that plane, against its rank-1
halves and the full-rank ceiling at the same site. Two sites: the end of layer 14, and the
pre-MLP residual (where H14 has just written and layer 14's own MLP has not yet read).

Needs `data_sharedrep.py` re-run with the `L14 u / v2 / uv plane / preMLP` conditions.

In [ ]:
# Grouped by site, so the rank-2 bar is read against the ceiling at *its own* site rather than
# against the other one's. rank is annotated on each bar: the claim is 2 of d_model directions.
SITES = [('end of layer 14', ['L14 u @n2', 'L14 v2 @n2', 'L14 uv plane @n2', 'L14 full @n2']),
         ('pre-MLP, layer 14', ['L14 uv plane preMLP @n2', 'L14 full preMLP @n2'])]
RANK = {'L14 u @n2': 1, 'L14 v2 @n2': 1, 'L14 uv plane @n2': 2,
        'L14 uv plane preMLP @n2': 2, 'L14 full @n2': M['d_model'],
        'L14 full preMLP @n2': M['d_model']}
UV_BAR_COL = {1: COL['h14'], 2: COL['das'], M['d_model']: COL['ceil']}

sites = [(t, [n for n in ns if f'cond_iia_{cond_key(n)}' in D]) for t, ns in SITES]
sites = [(t, ns) for t, ns in sites if ns]
if not sites:
    print('no L14-subspace conditions in this npz -- re-run data_sharedrep.py')
else:
    metrics = [('iia', 'IIA', (0, 1.12)), ('posrec', 'position recovery', None)]
    fig, axes = plt.subplots(len(metrics), len(sites), squeeze=False, sharey='row',
                             figsize=(2.7 * sum(len(ns) for _, ns in sites), 4.6 * len(metrics)),
                             gridspec_kw=dict(width_ratios=[len(ns) for _, ns in sites]))
    for col, (title, names_s) in enumerate(sites):
        for row, (metric, ylab, ylim) in enumerate(metrics):
            ax = axes[row][col]
            for i, nm in enumerate(names_s):
                v = D[f'cond_{metric}_{cond_key(nm)}']
                se = v.std() / np.sqrt(len(v))
                ax.bar(i, v.mean(), 0.66, yerr=se, capsize=4, color=UV_BAR_COL[RANK[nm]],
                       alpha=0.9, hatch=JOINT_HATCH if RANK[nm] == 2 else '',
                       edgecolor='white', linewidth=0)
                ax.text(i, v.mean() + se + 0.02 * np.ptp(ax.get_ylim()),
                        f'{v.mean():.2f}\nrank {RANK[nm]}', ha='center', va='bottom',
                        fontsize=TICK_FS - 2)
            if metric == 'posrec':
                ax.axhline(1.0, ls='--', color='0.6', lw=0.9)
            if ylim: ax.set_ylim(*ylim)
            ax.set_xticks(range(len(names_s)))
            ax.set_xticklabels([COND_SHORT[n] for n in names_s], fontsize=TICK_FS - 2)
            style_axes(ax, ylabel=ylab if col == 0 else None)
            if row == 0:
                ax.set_title(title, fontsize=TITLE_FS - 2)

    for _, names_s in sites:
        for nm in names_s:
            r, i = D[f'cond_posrec_{cond_key(nm)}'], D[f'cond_iia_{cond_key(nm)}']
            print(f'{nm:>26} (rank {RANK[nm]:>4}): position recovery {r.mean():6.3f} +/- '
                  f'{r.std() / np.sqrt(len(r)):.3f}   IIA {i.mean():.3f}')
    plt.tight_layout(); savefig(fig, f'{REGIME}_fig7_uv_plane_patching'); plt.show()

## Figure 8 — the same two 3D panels at the pre-MLP site

Panels (a) and (b) of the combined figure, redrawn on the residual stream **before** layer 14's
MLP at the second-number position: the point at which H14 has just written and the layer's own
MLP has not yet read. Same cloud, same directions, same probe recipe; only the read-out point
moves. `SITE = 'post'` re-runs the cell on the end-of-layer cloud for a like-for-like comparison.

Needs `data_sharedrep.py` re-run with the `resid14_pre` capture.

In [ ]:
SITE = 'pre'          # 'pre' = before layer 14's MLP, 'post' = the residual leaving layer 14
F8 = dict(arrow_lw=4.0, arrow_head=30, arrow_span=0.85, label_off=1.30,
          inset_box=[0.00, 0.66, 0.29, 0.27], box_zoom=1.18, cbar_pad=0.21, zlabel_pad=12)

# (cloud key, PCA-key tag, probe-key suffix, title) -- the two read-out points differ only here,
# because data_sharedrep.py runs one `pca_and_probes` recipe over both clouds.
SITE_SPEC = {'pre':  ('resid14_pre', '14pre', '_pre', "L14 residual before the MLP, @ $n_2$"),
             'post': ('resid14',     '14',    '',     "L14 residual leaving the layer, @ $n_2$")}
cloud_key, tag, sfx, site_title = SITE_SPEC[SITE]

if cloud_key not in D:
    print(f"'{cloud_key}' not in this npz -- re-run data_sharedrep.py")
else:
    cloud8 = D[cloud_key]
    xyz8, evr8 = xyz_for_plot(D[f'pca{tag}_coords']), D[f'pca{tag}_evr']
    PANELS8 = [(a_vals, '$y_1$', CMAP_A, ('h14_das', V1), (f'probe_logA{sfx}', r'$\mathbf{p}_1$'),
                '(a)'),
               (b_vals, '$y_2$', CMAP_B, ('l13_pc1b', V2), (f'probe_logB{sfx}', r'$\mathbf{p}_2$'),
                '(b)')]

    fig = plt.figure(figsize=(16.5, 7.2))
    for k, (values, vlab, cmap, main, probe, plet) in enumerate(PANELS8):
        ax = fig.add_subplot(1, 2, k + 1, projection='3d')
        ax.set_facecolor('none')
        for _axis in (ax.xaxis, ax.yaxis, ax.zaxis):
            _axis.pane.fill = False
            _axis.pane.set_edgecolor('none')
            _axis.line.set_color('0.65')
        ax.grid(False)
        sc = ax.scatter(xyz8[:, 0], xyz8[:, 1], xyz8[:, 2], c=values, cmap=cmap,
                        depthshade=False, zorder=3, s=12, alpha=0.7, edgecolors='none')

        centroid = xyz8.mean(axis=0)
        for (dkey, alab), acol in ((main, '#111111'), (probe, '#767676')):
            v3 = xyz_for_plot(D[f'{dkey}_in_pca{tag}'])
            # A direction's sign is arbitrary; its component's correlation with the number is not.
            if np.corrcoef(values, cloud8 @ D[f'{dkey}_dir'])[0, 1] < 0:
                v3 = -v3
            v3 = v3 / np.linalg.norm(v3)      # drawn as a direction; ||.|| is *_pca*_captured
            scale = F8['arrow_span'] * np.ptp(xyz8 @ v3)
            tail = centroid - v3 * scale * 0.5
            ax.add_artist(Arrow3D(tail, tail + v3 * scale, arrowstyle='-|>',
                                  mutation_scale=F8['arrow_head'], lw=F8['arrow_lw'], color=acol,
                                  shrinkA=0, shrinkB=0, zorder=12))
            _t = ax.text(*(tail + v3 * scale * F8['label_off']), alab, color=acol,
                         fontsize=LABEL_FS, fontweight='bold')
            _t.set_path_effects([pe.withStroke(linewidth=3, foreground='white')])
            print(f"  {dkey:>16}: norm captured in the 3D basis "
                  f"{float(D[f'{dkey}_pca{tag}_captured']):.3f}")

        pclab = lambda j: f'{PC_NAMES[j]} ({100 * evr8[j]:.0f}%)'
        ax.set_xlabel(pclab(PC_ORDER[0]), fontsize=LABEL_FS - 5)
        ax.set_ylabel(pclab(PC_ORDER[1]), fontsize=LABEL_FS - 5)
        ax.zaxis.set_rotate_label(False)
        ax.set_zlabel(pclab(PC_ORDER[2]), fontsize=LABEL_FS - 5, rotation=90)
        for _ax in (ax.xaxis, ax.yaxis):
            _ax.labelpad = 6
        ax.zaxis.labelpad = F8['zlabel_pad']
        ax.tick_params(labelsize=TICK_FS - 4, pad=0)
        ax.view_init(**VIEW)
        ax.set_box_aspect(None, zoom=F8['box_zoom'])
        cb = fig.colorbar(sc, ax=ax, shrink=0.5, pad=F8['cbar_pad'], aspect=18)
        cb.set_label(vlab, fontsize=LABEL_FS); cb.ax.tick_params(labelsize=TICK_FS - 2)
        cb.outline.set_visible(False)

        # Inset: the cloud on that panel's probe direction, against the number it predicts.
        axi = ax.inset_axes(F8['inset_box'])
        pvals = a_vals if plet == '(a)' else b_vals
        proj = D[f'{probe[0]}_comp'].astype(float)
        xu8, mu8, _ = aggregate(pvals, proj)
        axi.plot(xu8, mu8, 'o', ms=3, color=COL['das'] if plet == '(a)' else COL['full'],
                 alpha=0.9)
        p8, q8, r28 = fit_logx(pvals, proj)
        xf8 = np.linspace(pvals.min(), pvals.max(), 200)
        axi.plot(xf8, p8 * np.log(xf8) + q8, FIT_LS, color=FIT_COL, lw=1.8)
        style_inset(axi, '$y_1$' if plet == '(a)' else '$y_2$', 'probe comp.')
        ax.set_title(plet, fontsize=TITLE_FS + 2, fontweight='bold', loc='left')
        print(f"  probe {probe[0]}: held-out R2 = "
              f"{MJ[f'{probe[0]}_stats']['r2_test']:.3f}   log fit R2 = {r28:.3f}")

    print(f'{site_title}:  PCA EVR {np.round(evr8, 3).tolist()}')
    fig.suptitle(site_title, fontsize=TITLE_FS)
    fig.subplots_adjust(left=0.02, right=0.97, top=0.90, bottom=0.05, wspace=0.14)
    savefig(fig, f'{REGIME}_fig8_pca3_{SITE}mlp'); plt.show()

## Figure 9 — direction cosines with the pre-MLP probes

The panel-(c) heatmap redrawn with the $\log y_1$ / $\log y_2$ probes fitted on the residual
**before** layer 14's MLP — the point $\mathbf{v}_1$ is written into, and (Fig 7) the only site
where a rank-2 patch in span($\mathbf{v}_1$, $\mathbf{v}_2$) does anything. The post-MLP panel is
kept alongside as the reference; set `COS_SITES = ['pre']` for the pre panel alone.

Read the angle together with the probe's held-out $R^2$, printed under each panel: a more
orthogonal pair of worse probes is not a better result.

Needs `data_sharedrep.py` re-run with the pre-MLP probes in `cos_matrix`.

In [ ]:
COS_SITES = ['post', 'pre']    # which probe read-out points to draw, one panel each
COS_FS = dict(sym=17, name=12, cell=14, note=12, title=16)

SITE_LAB = {'post': ('', 'L14 out'), 'pre': (' pre', 'pre-MLP L14')}
have = [s for s in COS_SITES if f'probe log A{SITE_LAB[s][0]}' in M['cos_names']]
missing = [s for s in COS_SITES if s not in have]
if missing:
    print(f"no {missing} probes in this npz -- re-run data_sharedrep.py")


def cos_panel(ax, site):
    """One lower-triangular cosine heatmap, for the probes read out at `site`."""
    sfx, where = SITE_LAB[site]
    pA, pB = f'probe log A{sfx}', f'probe log B{sfx}'
    order = ['H14 DAS', pA, 'L13 PC1 @n2', pB]
    sym = {'H14 DAS': V1, pA: r'$\mathbf{p}_1$', 'L13 PC1 @n2': V2, pB: r'$\mathbf{p}_2$'}
    what = {'H14 DAS': '(L14 head 14)', pA: f'($\\log y_1$ probe,\n{where})',
            'L13 PC1 @n2': '(L13 @ $y_2$)', pB: f'($\\log y_2$ probe,\n{where})'}
    idx = [M['cos_names'].index(n) for n in order]
    C = D['cos_matrix'][np.ix_(idx, idx)]

    im = ax.imshow(np.where(np.tril(np.ones_like(C)) > 0, C, np.nan),
                   cmap=matplotlib.colormaps['RdBu_r'].with_extremes(bad='white'),
                   vmin=-1, vmax=1)
    ax.set_xticks(range(4)); ax.set_xticklabels([sym[n] for n in order], fontsize=COS_FS['sym'])
    ax.set_yticks(range(4))
    ax.set_yticklabels([f'{sym[n]}\n{what[n]}' for n in order], fontsize=COS_FS['name'])
    for i in range(4):
        for j in range(i + 1):
            ax.text(j, i, f'{C[i, j]:+.2f}', ha='center', va='center', fontsize=COS_FS['cell'],
                    color='white' if abs(C[i, j]) > 0.6 else 'black')
    # Same convention as panel (c): the causal pair boxed solid, the probe pair dashed as the
    # reference for what "not orthogonal" looks like at this d_model.
    pairs = ((order.index('H14 DAS'), order.index('L13 PC1 @n2'), '#111111', '-', 3.0),
             (order.index(pA), order.index(pB), '#767676', '--', 2.2))
    for i, j, ecol, els, elw in pairs:
        ax.add_patch(Rectangle((min(i, j) - 0.5, max(i, j) - 0.5), 1, 1, fill=False,
                               edgecolor=ecol, lw=elw, ls=els, zorder=6))
    deg = lambda i, j: np.degrees(np.arccos(np.clip(C[i, j], -1, 1)))
    r2 = (MJ[f'probe_logA{sfx.replace(" ", "_")}_stats']['r2_test'],
          MJ[f'probe_logB{sfx.replace(" ", "_")}_stats']['r2_test'])
    ax.set_xlabel(f'causal (solid) $\\angle$ = {deg(*pairs[0][:2]):.0f}$^\\circ$'
                  f'     probe (dashed) $\\angle$ = {deg(*pairs[1][:2]):.0f}$^\\circ$\n'
                  f'probe held-out $R^2$ = {r2[0]:.3f} / {r2[1]:.3f}',
                  fontsize=COS_FS['note'], labelpad=10)
    ax.set_title(f'probes read out at {where}', fontsize=COS_FS['title'])
    ax.grid(False); ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    print(f'{where:>12}:  causal {C[pairs[0][0], pairs[0][1]]:+.3f} '
          f'({deg(*pairs[0][:2]):.1f} deg)   probe {C[pairs[1][0], pairs[1][1]]:+.3f} '
          f'({deg(*pairs[1][:2]):.1f} deg)   probe R2 {r2[0]:.3f} / {r2[1]:.3f}')
    return im


if have:
    # Generous wspace: the row labels are two lines wide and would otherwise run into the
    # neighbouring panel's cells.
    fig, axes = plt.subplots(1, len(have), figsize=(7.4 * len(have), 6.2), squeeze=False,
                             gridspec_kw=dict(wspace=0.85))
    for ax, site in zip(axes[0], have):
        im = cos_panel(ax, site)
    cb = fig.colorbar(im, ax=axes[0].tolist(), shrink=0.72, pad=0.03)
    cb.set_label('cosine similarity', fontsize=COS_FS['title'] - 1)
    cb.ax.tick_params(labelsize=COS_FS['note'])
    print(f"chance |cos| = 1/sqrt(d) = {1 / np.sqrt(M['d_model']):.3f}")
    savefig(fig, f'{REGIME}_fig9_cosines_by_probe_site'); plt.show()

## Figure 10 — dose-response sweep of the span($\mathbf{v}_1$, $\mathbf{v}_2$) plane

The two pre-MLP coordinates are *set* to the values a chosen pair of numbers $(y_1^\*, y_2^\*)$
would put there, over a grid of pairs, and the model is asked which number it names.

$$
\mathrm{LD}(y_1^\*,y_2^\*)=\ell_{t(y_1^\*)}-\ell_{t(y_2^\*)},\qquad
\mathrm{LD}\;\overset{?}{=}\;g(y_1^\*)-h(y_2^\*)
$$

(a) fraction of prompts naming the implanted maximum · (b) the measured field, with its
$\mathrm{LD}=0$ boundary and the $y_1^\*=y_2^\*$ diagonal · (c) the best additive fit ·
(d) what the additive fit misses.

Needs `data_sharedrep.py` re-run with the sweep.

In [ ]:
SW_FS = dict(axis=15, tick=12, panel=19, cell=9, title=14)
SW_CMAP = matplotlib.colormaps['RdBu_r'].with_extremes(bad='0.92')

# What the sweep scores. The model only ever names a number that is in its prompt (99.5% of
# outputs here are one of the prompt's own two), so "did it say y1*?" is unanswerable -- the
# implanted numbers' tokens never win. What the implanted coordinates *can* decide is which
# **slot** the comparison picks, and that is exactly what the mechanism claims: u carries the
# first-slot number, v2 the second-slot number, and the read-out names the winner. So both the
# discrete and the continuous panels are scored on the prompt's own answer pair -- the same
# quantity position recovery is built from.
SWEEP_CASE = 'main'        # 'main' = meta['case'], 'alt' = meta['alt_case'] (ALT_CASE in the
                           # data script). The sweep sets absolute coordinates, so unlike the
                           # interchange conditions it asks the same question in either case.
_pre = '' if SWEEP_CASE == 'main' else 'alt_'
_case = M['case'] if SWEEP_CASE == 'main' else M.get('alt_case')
if f'{_pre}sweep_argmax' not in D:
    print(f"no '{_pre}sweep_*' arrays in this npz -- re-run data_sharedrep.py")
else:
    vals, am = D['sweep_values'], D[f'{_pre}sweep_argmax']
    G = len(vals)
    _a, _b = n1_slot_tokens(D, _case, _pre)
    t_n1, t_n2 = _a[None, :], _b[None, :]
    off = ~np.eye(G, dtype=bool)

    # (a) of the prompts that named one of their own two numbers, the share that named the
    # first-slot one. The prediction is 1 above the y1* = y2* diagonal and 0 below.
    P1 = np.full((G, G), np.nan)
    for i in range(G):
        for j in range(G):
            if i != j:
                n1, n2 = (am[i, j] == t_n1).sum(), (am[i, j] == t_n2).sum()
                P1[i, j] = n1 / max(n1 + n2, 1)

    has_ld = f'{_pre}sweep_ld_slot' in D
    panels = [('(a)', P1, 'P(names the first-slot number)', 'viridis', (0, 1), '.2f')]
    if has_ld:
        LDm = np.nanmean(D[f'{_pre}sweep_ld_slot'].astype(float), axis=2)
        # Two-way decomposition into a grand mean plus a row and a column effect -- exactly the
        # model LD = g(y1*) - h(y2*). The diagonal is missing, so the design is not orthogonal and
        # the two variance shares need not sum to 1; both are printed. A comparison thresholds a
        # contrast, so a large interaction is the expected outcome, not a failed test.
        gm = np.nanmean(LDm)
        ADD = gm + (np.nanmean(LDm, axis=1) - gm)[:, None] + (np.nanmean(LDm, axis=0) - gm)[None, :]
        ADD[np.diag_indices(G)] = np.nan
        INT = LDm - ADD
        tot = np.nansum((LDm - gm) ** 2)
        f_add, f_int = np.nansum((ADD - gm) ** 2) / tot, np.nansum(INT ** 2) / tot
        lim = np.nanmax(np.abs(LDm))
        panels += [('(b)', LDm, 'answer logit difference\n(first slot $-$ second slot)',
                    SW_CMAP, (-lim, lim), '.1f'),
                   ('(c)', ADD, 'additive fit  $g(y_1^*) - h(y_2^*)$', SW_CMAP, (-lim, lim), '.1f'),
                   ('(d)', INT, 'interaction (measured $-$ additive)', SW_CMAP, (-lim, lim), '.1f')]
    else:
        print(f"no '{_pre}sweep_ld_slot' -- re-run data_sharedrep.py")

    nr = 2 if len(panels) > 2 else 1
    nc = int(np.ceil(len(panels) / nr))
    fig, axes = plt.subplots(nr, nc, figsize=(6.8 * nc, 6.0 * nr), squeeze=False)
    for ax, (plet, Z, title, cmap, clim, fmt) in zip(axes.ravel(), panels):
        # The contour is the measured boundary. Contouring cannot cross the missing diagonal, so
        # for the contour only those cells are filled from their two in-row neighbours; the drawn
        # heatmap keeps them blank.
        Zc = Z.copy()
        for k in range(G):
            Zc[k, k] = np.mean([Zc[k, q] for q in (k - 1, k + 1) if 0 <= q < G])
        mid = 0.5 if plet == '(a)' else 0.0
        im = ax.imshow(np.ma.masked_invalid(Z), cmap=cmap, origin='lower',
                       vmin=clim[0], vmax=clim[1])
        for i in range(G):
            for j in range(G):
                if np.isfinite(Z[i, j]):
                    ax.text(j, i, format(Z[i, j], fmt), ha='center', va='center',
                            fontsize=SW_FS['cell'],
                            color='white' if abs(Z[i, j] - np.mean(clim)) > 0.62 * np.ptp(clim) / 2
                            else 'black')
        # y1* = y2* is where a "compare the two implanted numbers" read-out must switch.
        ax.plot([-0.5, G - 0.5], [-0.5, G - 0.5], color='0.35', lw=1.6, ls=':', zorder=4)
        if plet in ('(a)', '(b)'):
            ax.contour(np.arange(G), np.arange(G), Zc, levels=[mid], colors='#111111',
                       linewidths=2.6, zorder=5)
        ax.set_xticks(range(G)); ax.set_xticklabels(vals, fontsize=SW_FS['tick'])
        ax.set_yticks(range(G)); ax.set_yticklabels(vals, fontsize=SW_FS['tick'])
        ax.set_xlabel('$y_2^*$  (implanted along ' + V2 + ')', fontsize=SW_FS['axis'])
        ax.set_ylabel('$y_1^*$  (implanted along ' + V1 + ')', fontsize=SW_FS['axis'])
        ax.set_title(title, fontsize=SW_FS['title'])
        ax.grid(False); ax.tick_params(length=0)
        for s in ax.spines.values():
            s.set_visible(False)
        cb = fig.colorbar(im, ax=ax, shrink=0.82, pad=0.03)
        cb.ax.tick_params(labelsize=SW_FS['tick'] - 1); cb.outline.set_visible(False)
        ax.text(-0.16, 1.03, plet, transform=ax.transAxes, fontsize=SW_FS['panel'],
                fontweight='bold', ha='left', va='bottom')
    for ax in axes.ravel()[len(panels):]:
        ax.set_visible(False)

    upper = np.triu(np.ones((G, G), bool), 1).T & off      # y1* > y2*
    print(f'case {_case}: grid {G}x{G} on {am.shape[2]} prompts, values {vals.tolist()}')
    print(f'  outputs that are one of the prompt two numbers: '
          f'{np.mean([((am[i, j] == t_n1) | (am[i, j] == t_n2)).mean() for i, j in zip(*np.where(off))]):.3f}')
    print(f'  P(first slot): {np.nanmean(P1[upper]):.3f} when y1* > y2*   '
          f'{np.nanmean(P1[off & ~upper]):.3f} when y1* < y2*   '
          f'sign matches on {np.mean((P1[off] > 0.5) == (vals[:, None] > vals[None, :])[off]):.3f} of cells')
    if has_ld:
        print(f'  logit difference: {np.nanmean(LDm[upper]):+.2f} vs {np.nanmean(LDm[off & ~upper]):+.2f}   '
              f'sign matches on {np.mean((LDm[off] > 0) == (vals[:, None] > vals[None, :])[off]):.3f}')
        print(f'  variance of the field: additive {f_add:.3f}   interaction {f_int:.3f}   '
              f'(sum {f_add + f_int:.3f}; the missing diagonal makes the split non-orthogonal)')
    plt.tight_layout(); savefig(fig, f'{REGIME}_fig10_uv_plane_sweep_{SWEEP_CASE}'); plt.show()

## Figure 11 — the same conditions in both cases

The headline conditions run on `n1 larger` prompts (the case $\mathbf{v}_1$ was trained in) and
again on `n2 larger` prompts, **reusing the same directions** — nothing refitted.

Read the $\mathbf{v}_1$ bars with the counterfactual design in mind: `make_batch` varies only the
number at the winning slot, so under `n2 larger` the *first* number is the same in the clean and
corrupted prompts, H14's input is unchanged, and the $\mathbf{v}_1$ patch writes back a value it
already had. A near-zero bar there is a specificity control, not evidence that $\mathbf{v}_1$ is
inert when the second number wins — `docs/data_sharedrep.md` §6c has the batch design that would
test that.

Needs `data_sharedrep.py` re-run with `ALT_CASE` set.

In [ ]:
CASE_COL = {'main': COL['full'], 'alt': COL['das']}
C11_FS = dict(axis=16, tick=13, legend=13, val=12)

# The same six conditions the paper figure's panel (e) shows, so the two read together.
C11_CONDS = [('L13 full @n2', 'L13\nfull'), ('L13 PC1 @n2', V2), ('H14 DAS', V1),
             ('H14 DAS + L13 PC1 @n2', f'{V1} & {V2}'),
             ('L14 uv plane preMLP @n2', f'{V1},{V2}\nplane'), ('L14 full @n2', 'L14\nfull')]
C11_CONDS = [c for c in C11_CONDS if f'cond_iia_{cond_key(c[0])}' in D]
_has_alt = any(f'alt_cond_iia_{cond_key(c[0])}' in D for c in C11_CONDS)

if not _has_alt:
    print("no 'alt_cond_*' in this npz -- re-run data_sharedrep.py with ALT_CASE set")
else:
    ARMS11 = [('', M['case'], CASE_COL['main']), ('alt_', M.get('alt_case', 'alt'), CASE_COL['alt'])]
    metrics = [('iia', 'IIA', (0, 1.08)), ('posrec', 'position recovery', None)]
    fig, axes = plt.subplots(len(metrics), 1, figsize=(2.3 * len(C11_CONDS), 4.8 * len(metrics)),
                             squeeze=False, sharex=True)
    x11, W11 = np.arange(len(C11_CONDS)), 0.8 / len(ARMS11)
    for ax, (metric, ylab, ylim) in zip(axes[:, 0], metrics):
        for ai, (pre, lab, col) in enumerate(ARMS11):
            mus, ses = [], []
            for nm, _ in C11_CONDS:
                v = D.get(f'{pre}cond_{metric}_{cond_key(nm)}')
                mus.append(np.nan if v is None else v.mean())
                ses.append(np.nan if v is None else v.std() / np.sqrt(len(v)))
            xs = x11 + (ai - (len(ARMS11) - 1) / 2) * W11
            ax.bar(xs, mus, W11, yerr=ses, capsize=4, color=col, alpha=0.9, label=lab)
            for xi, mu, se in zip(xs, mus, ses):
                if np.isfinite(mu):
                    ax.text(xi, mu + se + 0.02 * (np.ptp(ylim) if ylim else 1.0), f'{mu:.2f}',
                            ha='center', va='bottom', fontsize=C11_FS['val'])
        if metric == 'posrec':
            ax.axhline(1.0, ls='--', color='0.6', lw=0.9)
        ax.axhline(0, color='0.6', lw=0.9)
        if ylim: ax.set_ylim(*ylim)
        style_axes(ax, ylabel=ylab)
        ax.set_ylabel(ylab, fontsize=C11_FS['axis']); ax.tick_params(labelsize=C11_FS['tick'])
    axes[0, 0].legend(fontsize=C11_FS['legend'], frameon=True, framealpha=0.92, edgecolor='0.8',
                      loc='upper left')
    axes[-1, 0].set_xticks(x11)
    axes[-1, 0].set_xticklabels([c[1] for c in C11_CONDS], fontsize=C11_FS['axis'] - 1)
    axes[-1, 0].set_xlabel('patched component, @ $n_2$', fontsize=C11_FS['axis'], labelpad=8)

    for nm, _ in C11_CONDS:
        row = []
        for pre, lab, _c in ARMS11:
            v = D.get(f'{pre}cond_iia_{cond_key(nm)}')
            r = D.get(f'{pre}cond_posrec_{cond_key(nm)}')
            row.append(f'{lab}: IIA {"  n/a" if v is None else f"{v.mean():.3f}"}  '
                       f'PR {"  n/a" if r is None else f"{r.mean():+.3f}"}')
        print(f'{nm:>26}   ' + '   |   '.join(row))
    plt.tight_layout(); savefig(fig, f'{REGIME}_fig11_conditions_both_cases'); plt.show()